# 📦 バッチ処理とパディング - 詳細技術ガイド
**NLP 実装の重要な基礎技術**

---

## 📚 目次
1. [なぜパディングが必要か](#1-なぜパディングが必要か)
2. [パディングの基本実装](#2-パディングの基本実装)
3. [Attention マスク](#3-attention-マスク)
4. [動的パディング (Dynamic Padding)](#4-動的パディング)
5. [パフォーマンス最適化](#5-パフォーマンス最適化)
6. [落とし穴と対処法](#6-落とし穴と対処法)

## 1. なぜパディングが必要か

In [ ]:
# 問題のデモ：テキストの長さはバラバラ
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

texts = [
    "AI",                                       # 短い
    "Machine learning is powerful",            # 中程度
    "Natural language processing enables computers to understand human text and language",  # 長い
]

print("各テキストのトークン数:")
for t in texts:
    ids = tokenizer.encode(t, add_special_tokens=True)
    print(f"  {len(ids):3d} tokens  |  {t[:50]}")

print()
print("行列にするには同じ長さが必要 → パディングで統一する")

In [ ]:
# パディングなし vs あり で変換を比較
print("=== padding=False (エラーになる) ===")
try:
    result = tokenizer(texts, return_tensors="pt")
    print("成功:", result["input_ids"].shape)
except ValueError as e:
    print(f"エラー発生: {e}")

print()
print("=== padding=True (最長に合わせる) ===")
result = tokenizer(texts, padding=True, return_tensors="pt")
print("成功:", result["input_ids"].shape)
print("Pad トークン ID:", tokenizer.pad_token_id)

print()
print("input_ids:")
for i, row in enumerate(result["input_ids"]):
    pads = (row == tokenizer.pad_token_id).sum().item()
    print(f"  行{i}  {row[:15].tolist()}...  [PAD数: {pads}]")

## 3. Attention マスク

パディングトークンは計算に含めないようにする

In [ ]:
# Attention マスクの役割
result = tokenizer(texts, padding=True, return_tensors="pt")

print("attention_mask:")
print("  1=実トークン  0=PAD")
for i, mask in enumerate(result["attention_mask"]):
    ones  = mask.sum().item()
    zeros = len(mask) - ones
    bar   = "▓" * int(ones) + "░" * int(zeros)
    print(f"  行{i}: [{bar}]  実={ones:3d} PAD={zeros:3d}")

# BERT に渡して確認
from transformers import AutoModel
bert = AutoModel.from_pretrained("bert-base-uncased")
with torch.no_grad():
    out = bert(**result)
print(f"\nBERT 出力 shape: {out.last_hidden_state.shape}")
print("(batch_size, seq_len, hidden_dim=768)")

## 4. 動的パディング (Dynamic Padding)

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# DataCollatorWithPadding: バッチ単位で最小限のパディング
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# データセットを模擬
raw_texts = [
    "Short text", "A bit longer sentence here",
    "The quick brown fox jumps", "AI", "Machine learning and deep learning are subfields of AI",
]
encodings = [tokenizer(t, truncation=True, max_length=64) for t in raw_texts]

# バッチ処理のシミュレーション
batch_size = 3
for i in range(0, len(encodings), batch_size):
    chunk = encodings[i:i+batch_size]
    batch = collator(chunk)
    print(f"バッチ {i//batch_size+1}: input_ids shape = {batch['input_ids'].shape}  "
          f"(動的に最小パディング)")

## 5. パフォーマンス最適化

In [ ]:
import time

# バッチサイズと速度の関係
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

def bench(batch_size, n_tokens=64):
    dummy_ids  = torch.ones(batch_size, n_tokens, dtype=torch.long)
    dummy_mask = torch.ones(batch_size, n_tokens, dtype=torch.long)
    start = time.time()
    with torch.no_grad():
        model(input_ids=dummy_ids, attention_mask=dummy_mask)
    return time.time() - start

sizes  = [1, 4, 8, 16]
times  = [bench(bs) for bs in sizes]
per_s  = [bs/t for bs, t in zip(sizes, times)]

print(f"{'BatchSize':<12} {'Time(s)':<12} {'Samples/s':<15}")
print("─" * 40)
for bs, t, ps in zip(sizes, times, per_s):
    print(f"{bs:<12} {t:<12.4f} {ps:<15.1f}")

## 6. 落とし穴と対処法

In [ ]:
# 落とし穴1: パディングを含む位置で損失計算 → ignore_index で対処
import torch.nn as nn

PAD_ID = 0

logits = torch.randn(2, 5, 1000)   # [batch=2, seq=5, vocab=1000]
labels = torch.tensor([[1, 2, 3,  PAD_ID, PAD_ID],
                       [4, 5, PAD_ID, PAD_ID, PAD_ID]])

# NG: PAD まで損失計算してしまう
loss_wrong = nn.CrossEntropyLoss()(logits.view(-1, 1000), labels.view(-1))

# OK: PAD を無視
loss_ok    = nn.CrossEntropyLoss(ignore_index=PAD_ID)(logits.view(-1, 1000), labels.view(-1))

print(f"PAD込み損失: {loss_wrong.item():.4f}  (大きめ → 間違い)")
print(f"PAD除外損失: {loss_ok.item():.4f}   (正しい)")

print()
print("落とし穴2: 左パディング vs 右パディング")
print("  GPT/Causal LM  → 左パディング (padding_side='left')")
print("  BERT/Encoder   → 右パディング (padding_side='right') ← デフォルト")

## ✅ 理解度チェック

- [ ] パディングなしで行列化できない理由が説明できる  
- [ ] Attention マスクが 0 の位置を計算から除外することが分かった  
- [ ] Dynamic Padding がメモリ節約になる理由が理解できた  
- [ ] `ignore_index=PAD_ID` を CrossEntropyLoss に渡す意味が分かった  

---
✅ 完了 → **[数学基礎ガイド](プロジェクトに必要な数学基礎.ipynb)** へ